[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# *Deep Agents*: planificación, archivos y subagentes.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 7: *Memoria y Deep Agents*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se usará `create_deep_agent` para una tarea larga: el agente planifica con una lista de tareas, escribe archivos en un sistema de archivos virtual y delega en subagentes. Se comparará con un agente de `create_agent`.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue el *notebook* `00_claves_de_api_y_secretos.ipynb`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue el notebook 00_claves_de_api_y_secretos: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from deepagents import create_deep_agent
from langchain.agents import create_agent

* La siguiente celda definirá las herramientas de políticas.

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

---
## 1. ¿Qué agrega un agente profundo?

| Capacidad | Para qué sirve | Herramienta interna |
|---|---|---|
| Lista de tareas | Planificar y no perder el hilo en tareas largas | `write_todos` |
| Sistema de archivos virtual | Guardar resultados intermedios fuera del contexto | `ls`, `read_file`, `write_file`, `edit_file` |
| Subagentes | Delegar subtareas con contexto aislado | `task` |
| Resumen automático | Evitar que el historial llene la ventana | *Middleware* de resumen |

> ⚠️ **Madurez.** `deepagents` aún no llega a la versión 1.0: sus versiones menores pueden romper compatibilidad. Por eso `requirements.txt` fija la versión exacta.

---
## 2. Un agente de análisis de políticas.

* La siguiente celda creará el agente profundo y le dará una tarea de análisis de varias políticas.

In [ ]:
analista = create_deep_agent(
    model=MODELO,
    tools=[listar_politicas, leer_politica],
    system_prompt=("Eres un analista de políticas internas de Andina Logística. Planifica antes de actuar, "
                   "guarda notas por política en archivos y escribe el informe final en informe.md."),
)
r = analista.invoke({"messages": [{"role": "user", "content":
    "Revisa todas las políticas y propón 3 mejoras para reducir tickets urgentes innecesarios."}]})
r["messages"][-1].text

* La siguiente celda mostrará la lista de tareas que el agente escribió para sí mismo.

In [ ]:
r.get("todos")

* La siguiente celda mostrará los archivos del sistema de archivos virtual y el informe.

In [ ]:
archivos = r.get("files", {})
print(list(archivos))
print(archivos.get("/informe.md", archivos.get("informe.md", "(sin informe)")))

**Observa:** los archivos viven en el **estado** del agente, no en tu disco. Revisa en *LangSmith* cuántas llamadas hizo y cuántas dedicó a planificar y a escribir archivos.

**💡 Ideas clave de la sección 2:**
* Un agente profundo es `create_agent` con *middleware* de planificación, archivos y subagentes.
* Conviene para tareas largas; para soporte de una pregunta es excesivo y caro.

### Ejercicio 1 · Subagente revisor.

Agrega al analista un subagente `revisor` (pasa `subagents=[{...}]` con `name`, `description` y `system_prompt`) que revise el informe y señale afirmaciones que no estén respaldadas por las políticas. Pide en el *prompt* del analista que use al revisor antes de terminar.

**🎯 Objetivo:** usar subagentes con contexto aislado. **Qué se espera aprender:** que delegar la revisión evita que el agente principal se califique a sí mismo.

* La siguiente celda muestra una solución posible.

In [ ]:
revisor = {
    "name": "revisor",
    "description": "Revisa un informe y señala afirmaciones no respaldadas por las políticas internas.",
    "system_prompt": "Eres un revisor estricto. Lee las políticas y marca cada afirmación del informe que no esté respaldada.",
    "tools": [listar_politicas, leer_politica],
}
analista_2 = create_deep_agent(model=MODELO, tools=[listar_politicas, leer_politica], subagents=[revisor],
    system_prompt=("Eres un analista de políticas internas. Escribe el informe en informe.md y, antes de "
                   "terminar, pide al subagente revisor que lo revise y corrige lo que señale."))
r2 = analista_2.invoke({"messages": [{"role": "user", "content":
    "Revisa todas las políticas y propón 3 mejoras para reducir tickets urgentes innecesarios."}]})
[tc["args"].get("subagent_type") for m in r2["messages"] if m.type == "ai" for tc in m.tool_calls if tc["name"] == "task"]

### Ejercicio 2 · Caso de diseño: ¿cuál herramienta?

Para cada caso, elige entre `create_agent`, un grafo de *LangGraph* o `create_deep_agent`, y justifica:
1. Un asistente que responde dudas de políticas en el chat interno.
2. Un agente que cada lunes investiga a 10 competidores y redacta un informe de 5 páginas.
3. Un flujo de aprobación de gastos con reglas fijas y un paso de revisión humana.

**🎯 Objetivo:** elegir el nivel de abstracción correcto. **Qué se espera aprender:** que cada herramienta resuelve un tipo distinto de problema.

**✅ Respuesta sugerida:**

1. **`create_agent`**: preguntas cortas, pocas herramientas, baja latencia. 2. **`create_deep_agent`**: tarea larga que se beneficia de planificar, guardar notas por competidor y delegar en subagentes. 3. **Grafo de *LangGraph***: el flujo es conocido y debe ser auditable; el modelo solo participa en pasos concretos y la aprobación humana es un nodo con `interrupt`.

---
## ✅ Cierre.

**Práctica:** terminar el ejercicio 2 y revisar la traza del agente profundo.

**Siguiente *notebook*:** `13_evaluacion_y_observabilidad.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>